# Installing packages

In [1]:
!pip install torch pandas nltk sentencepiece bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


# Importing Packages

In [2]:
import os
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as neural_net
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import sentencepiece as sent_pi
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score

import warnings
warnings.filterwarnings('ignore')

## Setting Device

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Loading Dataset

In [4]:
def dataset_load(sa_path, en_path):
    sa_df = pd.read_csv(sa_path)
    en_df = pd.read_csv(en_path)
    merged_df = pd.merge(sa_df, en_df, on="Source_id")
    return merged_df[["Source_id", "Sentence_sa", "Sentence_en"]]

train_df = dataset_load("data/train_sa_10000.csv", "data/train_en_10000.csv")
dev_df = dataset_load("data/dev_sa_1000.csv", "data/dev_en_1000.csv")
test_df = dataset_load("data/test_sa_1000.csv", "data/test_en_1000.csv")
print(f"Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")

Train: 10000, Dev: 1000, Test: 1000


# SentencePiece Tokenizer training

In [5]:
def train_sentence_piece(sentences, model_prefix, vocab_size=8000):
  with open(f"{model_prefix}_corpus.txt", "w", encoding="utf-8") as f:
    for sentence in sentences:
      f.write(str(sentence) + "\n")

  sent_pi.SentencePieceTrainer.train(
    input=f"{model_prefix}_corpus.txt",
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=1.0,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
  )
  os.remove(f"{model_prefix}_corpus.txt")
  return sent_pi.SentencePieceProcessor(model_file=f"{model_prefix}.model")

## Preparing Sentences


In [6]:
source_sentences = train_df["Sentence_sa"].tolist() + dev_df["Sentence_sa"].tolist() + test_df["Sentence_sa"].tolist()
target_sentences = train_df["Sentence_en"].tolist() + dev_df["Sentence_en"].tolist() + test_df["Sentence_en"].tolist()

source_sp = train_sentence_piece(source_sentences, "spm_source", vocab_size=8000)
target_sp = train_sentence_piece(target_sentences, "spm_target", vocab_size=5000)

source_pad_id = source_sp.pad_id()
target_pad_id = target_sp.pad_id()
target_bos_id = target_sp.bos_id()
target_eos_id = target_sp.eos_id()

print(f"Source vocab size: {source_sp.vocab_size()}")
print(f"Target vocab size: {target_sp.vocab_size()}")

Source vocab size: 8000
Target vocab size: 5000


# Dataset Loader and Dataset

## Translation Dataset

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, data_frame, source_sp, target_sp, max_length=50):
        self.data_frame = data_frame
        self.source_sp = source_sp
        self.target_sp = target_sp
        self.max_length = max_length

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, index):
        source_text = self.data_frame.iloc[index]["Sentence_sa"]
        target_text = self.data_frame.iloc[index]["Sentence_en"]
        source_indexes = self.source_sp.encode(source_text, out_type=int)
        target_indexes = [self.target_sp.bos_id()] + self.target_sp.encode(target_text, out_type=int) + [self.target_sp.eos_id()]
        source_indexes = source_indexes[:self.max_length]
        target_indexes = target_indexes[:self.max_length]
        return torch.tensor(source_indexes, dtype=torch.long), torch.tensor(target_indexes, dtype=torch.long)

## Creating a collate function

In [8]:
def collate(batch):
  source_batch, target_batch = zip(*batch)
  source_lengths = torch.tensor([len(s) for s in source_batch], dtype=torch.long)
  target_lengths = torch.tensor([len(t) for t in target_batch], dtype=torch.long)
  source_padded = pad_sequence(source_batch, batch_first=True, padding_value=source_sp.pad_id())
  target_padded = pad_sequence(target_batch, batch_first=True, padding_value=source_sp.pad_id())
  return source_padded, source_lengths, target_padded, target_lengths

## Using TranslationDataset Class

In [9]:
batch_size = 32
train_dataset = TranslationDataset(train_df, source_sp, target_sp)
dev_dataset = TranslationDataset(dev_df, source_sp, target_sp)
test_dataset = TranslationDataset(test_df, source_sp, target_sp)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)

# Model Definition

Here we will create model. We will be using following details for this purpose:-

*   Encoder
*   Attention
*   Decoder
*   Seq2Seq




## Encoder

We will be using embedding with LSTM

### Creating class for Encoder

In [12]:
class Encoder(neural_net.Module):
  def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
    super().__init__()
    self.embedding = neural_net.Embedding(vocab_size, embed_size, padding_idx=source_sp.pad_id())
    self.lstm = neural_net.LSTM(embed_size, hidden_size, num_layers, dropout=dropout, batch_first=True, bidirectional=True)
    self.dropout = neural_net.Dropout(dropout)

  def forward(self, x, lengths):
    embedded = self.dropout(self.embedding(x))
    packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
    outputs, (hidden, cell) = self.lstm(packed)
    outputs, _ = pad_packed_sequence(outputs, batch_first=True)
    hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
    cell   = torch.cat((cell[-2],   cell[-1]),   dim=1)
    return outputs, hidden.unsqueeze(0), cell.unsqueeze(0)



## Attention

### Implementing LuongAttention

In [13]:
class LuongAttention(neural_net.Module):
  def __init__(self, hidden_size):
    super().__init__()
    self.hidden_size = hidden_size

  def forward(self, query, values, mask=None):
    sources = torch.bmm(query, values.transpose(1, 2))
    if mask is not None:
      sources = sources.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(sources, dim=-1)
    context = torch.bmm(attention_weights, values)
    return context, attention_weights


### Decoder

In [14]:
class Decoder(neural_net.Module):
  def __init__(self, vocab_size, embed_size, encoder_hidden_size, decoder_hidden_size, num_layers, dropout, attention):
    super().__init__()
    self.embedding = neural_net.Embedding(vocab_size, embed_size, padding_idx=target_sp.pad_id())
    self.lstm = neural_net.LSTM(embed_size + encoder_hidden_size,  decoder_hidden_size, num_layers, dropout=dropout, batch_first=True)
    self.attention = attention
    self.fc = neural_net.Linear(decoder_hidden_size + encoder_hidden_size, vocab_size)
    self.dropout = neural_net.Dropout(dropout)

  def forward(self, x, encoder_outputs, hidden, cell, mask=None):
    embedded = self.dropout(self.embedding(x))
    context, attention_weights = self.attention(hidden[-1].unsqueeze(1), encoder_outputs, mask)
    rnn_input = torch.cat((embedded, context), dim=2)
    output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))
    combined = torch.cat((output, context), dim=2)
    prediction = self.fc(combined)
    return prediction, hidden, cell, attention_weights

## Sequence to Sequence Model (Seq2Seq Model)

In [15]:
class Seq2SeqModel(neural_net.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder


  def forward(self, source, source_length, target, forcing_ratio=0.5):
    batch_size, target_length = target.shape[0], target.shape[1]
    target_vocab_size = self.decoder.fc.out_features
    outputs = torch.zeros(batch_size, target_length, target_vocab_size).to(device)
    encoder_outputs, hidden, cell = self.encoder(source, source_length)
    x = target[:, 0]
    for t in range(1, target_length):
      prediction, hidden, cell, _ = self.decoder(x.unsqueeze(1), encoder_outputs, hidden, cell)
      outputs[:, t, :] = prediction.squeeze(1)
      teacher_force = torch.rand(1).item() < forcing_ratio
      top1 = prediction.argmax(2).squeeze(1)
      x = target[:, t] if teacher_force else top1
    return outputs

In [16]:
embed_size = 256
hidden_size = 512
num_layers = 2
dropout = 0.3

encoder = Encoder(
    source_sp.vocab_size(),
    embed_size,
    hidden_size,
    num_layers,
    dropout
  )
encoder_hidden_size = 2 * hidden_size
decoder_hidden_size = encoder_hidden_size
decoder = Decoder(
    vocab_size=target_sp.vocab_size(),
    embed_size=embed_size,
    encoder_hidden_size=encoder_hidden_size,
    decoder_hidden_size=decoder_hidden_size,
    num_layers=1,
    dropout=dropout,
    attention=LuongAttention(encoder_hidden_size)
  )
model = Seq2SeqModel(encoder, decoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = neural_net.CrossEntropyLoss(ignore_index=target_sp.pad_id())
schedular = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

def count_parameters(model):
  return sum(param.numel() for param in model.parameters() if param.requires_grad)

print(f"Total parameters to train: {count_parameters(model):,}")



Total parameters to train: 32,471,944


# Training Loop

### Training epochs

In [22]:
def train_epoch(model, loader, optimizer, criterion, forcing_ratio=0.5):
  model.train()
  total_loss = 0
  for batch in loader:
    source, source_length, target, _ = batch
    source = source.to(device)
    target = target.to(device)
    optimizer.zero_grad()
    output = model(source, source_length, target, forcing_ratio)
    output = output[:, 1:, :].reshape(-1, output.shape[-1])
    target = target[:, 1:].reshape(-1)
    loss = criterion(output, target)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    total_loss += loss.item()
  return total_loss / len(loader)

In [23]:
def greedy_decode(model, source, source_length, max_length=50):
  model.eval()
  with torch.no_grad():
    encoder_outputs, hidden, cell = model.encoder(source, source_length)
    x = torch.tensor([[target_sp.bos_id()]] * source.shape[0], device=device)
    predictions = torch.zeros(source.shape[0], max_length, dtype=torch.long, device=device)
    for t in range(max_length):
      prediction, hidden, cell, _ = model.decoder(x, encoder_outputs, hidden, cell)
      top1 = prediction.argmax(2).squeeze(1)
      predictions[:, t] = top1
      x = top1.unsqueeze(1)
      if (top1 == target_sp.eos_id()).all():
        break
    return predictions.cpu()

In [24]:
def evaluate_bleu(model, loader, max_length=50):
  refs, hyps = [], []
  for batch in loader:
    source, source_length, target, _ = batch
    source = source.to(device)
    prediction_ids = greedy_decode(model, source, source_length, max_length)
    for i in range(source.size(0)):
      reference_tokens = target_sp.decode(target[i].tolist()).split()
      hyp_tokens = target_sp.decode(prediction_ids[i].tolist()).split()
      if "<eos>" in hyp_tokens:
        hyp_tokens = hyp_tokens[:hyp_tokens.index('<eos>')]
      if "<eos>" in reference_tokens:
        reference_tokens = reference_tokens[:reference_tokens.index('<eos>')]
      refs.append([reference_tokens])
      hyps.append(hyp_tokens)
  return corpus_bleu(refs, hyps)


### Training Loop

In [25]:
best_blue = 0.0
epochs = 20
for epoch in range(1, epochs+1):
  train_loss = train_epoch(model, train_loader, optimizer, criterion, forcing_ratio=0.5)
  dev_bleu = evaluate_bleu(model, dev_loader)
  schedular.step(dev_bleu)
  print(f"Epoch {epoch:2d}: Train Loss:={train_loss:.4f}, dev BLEU:={dev_bleu:.4f}")

  if dev_bleu > best_blue:
    best_blue = dev_bleu
    torch.save(model.state_dict(), "best_model.pt")
    print("====== NEW BEST MODEL SAVED ======")

model.load_state_dict(torch.load("best_model.pt", map_location=device))


NameError: name 'scheduler' is not defined

# Beam Search Implementation

## Creating beam search function

In [ ]:
def beam_search_decode(model, source, source_len, beam_size=4, max_length=50):
  model.eval()
  batch_size = source.shape[0]
  all_hypothesis = []
  with torch.no_grad():
    encoder_outputs, hidden, cell = model.encoder(source, source_len)
    encoder_outputs = encoder_outputs.repeat_interleave(beam_size, dim=0)
    hidden = hidden.repeat_interleave(beam_size, dim=1)
    cell = cell.repeat_interleave(beam_size, dim=1)

    x = torch.tensor([[target_sp.bos_id()]] * (batch_size * beam_size), device=device)
    sequences = torch.full((batch_size, beam_size, 1), target_sp.bos_id(), dtype=torch.long, device=device)
    scores = torch.zeros(batch_size, beam_size, dtype=torch.bool, device=device)

  for _ in range(max_length):
    predictions, hidden, cell, _ = model.decoder(x, encoder_outputs, hidden, cell)
    log_probs = F.log_softmax(predictions.squeeze(1), dim=-1)
    for i in range(batch_size * beam_size):
      if finished[i]:
        continue
      scores_i = scores[i] + log_probs[i]
      top_scores, top_indices = torch.topk(scores_i, k=beam_size)
  pass


In [ ]:
def beam_search_for_single_source_sentence(model, source_tensor, beam_size=4, max_length=50):
  model.eval()
  with torch.no_grad():
    source_tensor = source_tensor.unsqueeze(0).to(device)
    source_length = torch.tensor([len(source_tensor[0])], device=device)
    encoder_outputs, hidden, cell = model.encoder(source_tensor, source_length)

    encoder_outputs = encoder_outputs.repeat(beam_size, 1, 1)
    hidden = hidden.repeat(1, beam_size, 1)
    cell = cell.repeat(1, beam_size, 1)

    sequences = torch.full((beam_size, 1), target_sp.bos_id(), dtype=torch.long, device=device)
    scores = torch.zeros(beam_size, device=device)
    finished = torch.zeros(beam_size, dtype=torch.bool, device=device)

    for _ in range(max_length):
      if finished.all():
        break
      x = sequences[:, -1].unsqueeze(1)
      predictions, hidden, cell, _ = model.decoder(x, encoder_outputs, hidden, cell)
      log_probs = F.log_softmax(predictions.squeeze(1), dim=-1)

      all_candidates = []
      for i in range(beam_size):
        if finished[i]:
          all_candidates.append((scores[i], sequences[i].tolist()))
          continue
        for j in range(beam_size):
          source = source[i] + log_probs[i][j]
          token = j
          sequence = sequences[i].tolist() + [token]
          all_candidates.append((source, sequence))

      all_candidates.sort(key=lambda x: x[0], reverse=True)
      best = all_candidates[:beam_size]
      for i, (score, sequence) in enumerate(best):
        sequences[i] = torch.tensor(sequence, device=device)
        scores[i] = score
        if sequence[-1] == target_sp.eos_id():
          finished[i] = True

    best_index = scores.argmax()
    return sequences[best_index].cpu().tolist()


# Validation And Evaluation

## Validation

In [ ]:
def generate_submission(model, loader, decode_function="beam"):
  model.eval()
  predictions = []
  source_ids = []
  for batch in loader:
    source, source_length, target, _ = batch
    source = source.to(device)
    source_ids.extend(batch[3].tolist())

In [ ]:
model.eval()
predictions = []
for index, row in test_df.iterrows():
  source_text = row["Sentence_sa"]
  source_ids = source_sp.encode(source_text, out_type=int)[:50]
  source_tensor = torch.tensor(source_ids, device=device).unsqueeze(0)
  source_len = torch.tensor([len(source_ids)], device=device)

  with torch.no_grad():
    encoder_outputs, hidden, cell = model.encoder(source_tensor, source_len)
    x = torch.tensor([[target_sp.bos_id()]], device=device)
    prediction_tokens = []
    for _ in range(50):
      prediction, hidden, cell, _ = model.decoder(x, encoder_outputs, hidden, cell)
      top1 = prediction.argmax(2).squeeze(1)
      token = top1.item()
      if token == target_sp.eos_id():
        break
      prediction_tokens.append(token)
      x = top1.unsqueeze(1)
    prediction_text = target_sp.decode(prediction_tokens)
  predictions.append({"Source_id": row["Source_id"], "Sentence_en": prediction_text})

submission_df = pd.DataFrame(predictions)
submission_df.to_csv("submission.csv", index=False, encoding="utf-8")
print("submission.csv file created!")

## Evaluation

In [ ]:
gold_test = test_df["Sentence_en"].tolist()
pred_test = submission_df["Sentence_en"].tolist()

refs = [[ref.split()] for ref in gold_test]
hyps = [pred.split() for pred in pred_test]
bleu_score = corpus_bleu(gold_test, pred_test)
print(f"BLEU Score: {bleu_score*100:.2f}")

P, R, F1 = bert_score(pred_test, gold_test, lang="en", rescale_with_baseline=True)
bert_f1 = F1.mean().item()
print(f"Test BERT score F1: {bert_f1:.4f}")

start = time.time()
print(f"Inference time for test set: {time.time() - start:.2f} seconds")
print(f"Total parameters: {count_parameters(model):,}")